# Genizah v2.1b (DISPATCH FIX) — accurate line/word boxes via direct grounding (data lever on v2.0a)

**Supersedes `genizah_v21` (W&B `js3ku3tw`, 2026-09-11), which trained image-blind:** with a streamed
(iterable) train set Accelerate defaulted to `DataLoaderDispatcher`, which slices every batch tensor along
dim 0 to the batch size — Qwen3-VL's packed `pixel_values` `[25728, 1536]` reached the model as `[1, 1536]`.
This notebook sets `accelerator_config={"dispatch_batches": False}`, gates the PREPARED train dataloader
before the first step, and writes to fresh destinations (`v21b-ckpt`, `outputs_v21b`, W&B `genizah_v21b`).
Never resume from `v21-ckpt`. Post-mortem: `docs/postmortem_v21_dispatch_truncation.md`.

ONE variable vs v2.0a: the grounding DATA. `genizah_ktiv_v3` adds six direct-grounding
families built from KTIV word geometry — `locate_word`, `read_box_word`, `line_index`,
`line_of_phrase`, `grounded_detect` (bbox before text) and `grounded_crop` (per-column
band crops with re-normalized boxes) — and the grounding share rises from 8% to 20%.
Everything else (install triplet, resolution contract 6.5/7 MP, LoRA config, merger
FROZEN, schedule, hub checkpointing) is v2.0a's. Warm start = v2.0a step 1800.

Why: v2.0a's line boxes were a layout prior (template x-range repeated down the page,
58% on the right line). See `docs/v21_grounding_boxes_design.md`; gate =
`grounding_eval/box_quality.py` + grounding trio + religious-140/PGP-131 no-regression.

Data is STREAMED per family from the hub (one `train_<task>` split each): the dataset is
~70 GB because every row embeds its page JPEG, and Colab's disk cannot hold the parquet
download and the arrow cache at once. Only the sampled share is ever downloaded.


In [ ]:
# Cell 1 — installs + env (PINNED: the exact triplet audited 2026-08-09;
# unpinned installs float and transformers 5.x would invalidate the vision-path
# analysis AND the 108-tensor count)
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["WANDB_PROJECT"] = "qwen-hebrew-finetune"
%pip install -q "unsloth[colab-new]==2026.8.9" "unsloth_zoo==2026.8.6" "transformers==4.57.6" hf_transfer wandb

from importlib.metadata import version
assert version("unsloth_zoo") == "2026.8.6", f"unsloth_zoo drifted: {version('unsloth_zoo')}"
assert version("transformers") == "4.57.6", f"transformers drifted: {version('transformers')}"

import torch
assert torch.cuda.is_available(), "No GPU — switch runtime to A100"

In [ ]:
# Cell 2 — data: genizah_ktiv_v3 (transcription + direct grounding, STREAMED per family) + clean_v2 + synth3 + talmud replay
from google.colab import userdata
from huggingface_hub import login, snapshot_download
from datasets import load_dataset, load_dataset_builder, concatenate_datasets

login(token=userdata.get("HF_TOKEN"))

KTIV3_REPO = "isaacmg/genizah_ktiv_v3"
KTIV3_REVISION = "45fe477baf11029e89014633dbcad974591e3363"  # 60,001 rows: v2 families + 6 direct-grounding families; per-task train splits
GENIZAH_REPO = "isaacmg/genizah_clean_v2"
GENIZAH_REVISION = "57366ad378946918731ad0012d699acc7d9ed31c"  # 1,055/56 repaired+screened
SYNTH3_REPO = "isaacmg/synthetic_hebrew_v3"
SYNTH3_REVISION = "59abcf7c30fb6753b9df89f0a099b07a68059013"  # 12 faces, probe-weighted drills
TALMUD_REPO = "isaacmg/talmud_finetune_v2"

for _sha in (KTIV3_REVISION, GENIZAH_REVISION, SYNTH3_REVISION):
    assert len(_sha) == 40, f"unpinned revision: {_sha!r}"

# v2.1 hub layout: one train split per task family (train_<task>) + one val split.
KTIV_TRANSCRIBE = ("fragment_transcribe", "region_transcribe", "section_transcribe", "line_transcribe")
GROUNDING_V20 = ("locate", "read_box", "layout_qa", "grounded_page")
GROUNDING_V21 = ("locate_word", "read_box_word", "line_index", "line_of_phrase",
                 "grounded_detect", "grounded_crop")
GROUNDING_TASKS = GROUNDING_V20 + GROUNDING_V21
_splits = load_dataset_builder(KTIV3_REPO, revision=KTIV3_REVISION).info.splits
N_ROWS = {name: _splits[f"train_{name}"].num_examples for name in KTIV_TRANSCRIBE + GROUNDING_TASKS}
print("genizah_ktiv_v3 train rows per family:", N_ROWS)

def stream(name, seed=3407, buffer=512):
    """Streamed + buffer-shuffled iterable over one train family (reads only its own shards)."""
    return load_dataset(KTIV3_REPO, split=f"train_{name}", revision=KTIV3_REVISION,
                        streaming=True).shuffle(seed=seed, buffer_size=buffer)

def as_stream(ds, seed=3407):
    """Small map-style source -> shuffled iterable (interleave needs one kind of dataset)."""
    return ds.shuffle(seed=seed).to_iterable_dataset(num_shards=4)

ktiv_pages = stream("fragment_transcribe")
ktiv_regions = stream("region_transcribe")
ktiv_sections = stream("section_transcribe")
ktiv_lines = stream("line_transcribe")
grounding_streams = {name: stream(name) for name in GROUNDING_TASKS}
# map-style val (~3.5 GB), fetched by data_files so ONLY the val shards download —
# load_dataset(split="val") on this 15-split repo would pull the whole 77 GB config first.
ktiv3_val = load_dataset(KTIV3_REPO, data_files={"val": "data/val-*.parquet"},
                         split="val", revision=KTIV3_REVISION,
                         verification_mode="no_checks")  # only val is present; skip all-splits check

genizah = load_dataset(GENIZAH_REPO, split="train", revision=GENIZAH_REVISION)
genizah_val = load_dataset(GENIZAH_REPO, split="val", revision=GENIZAH_REVISION)
synth3 = load_dataset(SYNTH3_REPO, split="train", revision=SYNTH3_REVISION)
synth3_eval = load_dataset(SYNTH3_REPO, split="eval", revision=SYNTH3_REVISION)
talmud = load_dataset(TALMUD_REPO, split="train")
talmud_val = load_dataset(TALMUD_REPO, split="val")

# Talmud replay (forgetting protection; same v16-lesson exclusions as v18/v19/v20)
crops = talmud.filter(lambda t: t == "crop_transcribe", input_columns="task")
pages = talmud.filter(lambda t: t == "page_extract", input_columns="task")
gemara_crops = crops.filter(lambda s: s == "gemara", input_columns="section")
pages_small = pages.filter(
    lambda s: s in ("rashi", "tosafot"), input_columns="section")
_n_before = len(pages_small)
pages_small = pages_small.filter(
    lambda st: st.split("_")[0] not in ("16", "05"), input_columns="stem")
print(f"pages_small: {_n_before} -> {len(pages_small)} after vol 16/05 exclusion")
assert len(pages_small) < _n_before, "vol 16/05 exclusion filtered nothing"

print(f"ktiv3: pages={N_ROWS['fragment_transcribe']} regions={N_ROWS['region_transcribe']} "
      f"crops={N_ROWS['section_transcribe'] + N_ROWS['line_transcribe']} "
      f"grounding={sum(N_ROWS[n] for n in GROUNDING_TASKS)} val={len(ktiv3_val)} | "
      f"genizah={len(genizah)} synth3={len(synth3)} "
      f"pages_small={len(pages_small)} gemara_crops={len(gemara_crops)}")
assert N_ROWS["fragment_transcribe"] > 3000 and N_ROWS["region_transcribe"] > 4000
assert N_ROWS["section_transcribe"] + N_ROWS["line_transcribe"] > 8000
assert sum(N_ROWS[n] for n in GROUNDING_TASKS) > 30000, "v2.1 grounding families missing"
assert N_ROWS["locate_word"] > 9000 and N_ROWS["line_index"] > 6000 and N_ROWS["grounded_crop"] > 3000
assert N_ROWS["grounded_detect"] > 2500 and len(ktiv3_val) >= 2500
assert len(genizah) > 1000 and len(genizah_val) >= 50
assert len(synth3) > 10000 and len(synth3_eval) >= 800

# Label hygiene (transcription answers only — grounding answers are JSON/lines)
_ktiv_sample = [r["answer"] for r in ktiv_pages.take(200)]
for _name, _sample in (("genizah", genizah.shuffle(seed=0).select(range(200))["answer"]),
                       ("ktiv", _ktiv_sample)):
    assert all("␣" not in a for a in _sample), f"{_name}: internal gap token leaked"
    assert all(not any(ch in a for ch in "&#$_{}<>\\") for a in _sample), f"{_name}: mojibake leaked"
    assert any("[...]" in a for a in _sample), f"{_name}: expected damage-gap markers"
_q = [r["question"] for r in ktiv_regions.take(50)]
assert all("Transcribe ONLY" in q for q in _q), "region rows lost their restriction"

# Grounding hygiene: valid 0-1000 JSON, self-describing prompts, bbox-FIRST in the
# detect/crop families (the v2.1 lever), text-first still valid in grounded_page
import json as _json
def _ok_box(b):
    return len(b) == 4 and all(0 <= v <= 1000 for v in b) and b[2] > b[0] and b[3] > b[1]
for _fam in ("locate", "locate_word"):
    for _r in grounding_streams[_fam].take(50):
        assert _ok_box(_json.loads(_r["answer"])["bbox_2d"]), f"bad {_fam} box"
        assert "0-1000" in _r["question"]
for _fam in ("line_index", "line_of_phrase"):
    for _r in grounding_streams[_fam].take(30):
        _o = _json.loads(_r["answer"])
        assert _ok_box(_o["bbox_2d"]) and _o["text"].strip(), f"bad {_fam} payload"
    if _fam == "line_index":
        assert all("counting from the" in r["question"] for r in grounding_streams[_fam].take(10))
for _fam in ("grounded_detect", "grounded_crop"):
    for _r in grounding_streams[_fam].take(10):
        _arr = _json.loads(_r["answer"])
        assert isinstance(_arr, list) and _arr and list(_arr[0].keys()) == ["bbox_2d", "text"], \
            f"{_fam} must be bbox-first"
        assert all(_ok_box(e["bbox_2d"]) and e["text"] for e in _arr), f"bad {_fam} payload"
for _r in grounding_streams["grounded_page"].take(10):
    _arr = _json.loads(_r["answer"])
    assert isinstance(_arr, list) and all(_ok_box(e["bbox_2d"]) and e["text"] for e in _arr)
for _fam in ("read_box", "read_box_word"):
    assert all("bbox_2d = [" in r["question"] and "0-1000" in r["question"]
               for r in grounding_streams[_fam].take(30))
print("grounding hygiene OK: v2.0 families + locate_word/read_box_word/line_index/line_of_phrase/"
      "grounded_detect/grounded_crop validated (bbox-first where required)")


In [ ]:
# Cell 3 — model at page resolution + THE MERGER KNOB + flagship warm start
from unsloth import FastVisionModel
from unsloth_zoo.peft_utils import get_peft_regex
from transformers import AutoImageProcessor

MAX_SEQ = 12288
# Global 6.5MP floor (train/infer resolution contract, ships in the export).
MIN_PIX = 6_500_000
MAX_PIX = 7_000_000

# ⬅️ THE DECISION KNOB — set per the v1.9c verdict:
#   "frozen" = v1.9a config (default, safe)   "full" = v1.9c full-weight merger
#   "lora"   = v1.9b merger LoRA (two-sided at r16 per the ablation; avoid)
MERGER_MODE = "frozen"
assert MERGER_MODE in ("frozen", "lora", "full")

# Warm start: v2.0a step 1800 (the v2.1 data lever sits on top of the grounded model). For a v1.9c warm start use
# WARM_CKPT_REPO = "isaacmg/qwen3-vl-8b-hebrew-v19c-ckpt" with the SHA of the
# commit "Training in progress, step 700, checkpoint" — and MERGER_MODE="full".
WARM_CKPT_REPO = "isaacmg/qwen3-vl-8b-hebrew-v20a-ckpt"
WARM_REVISION = "af9df6a0ad4743bc8493a7cf0c14bc3b4b796dd8"  # v2.0a step 1800 (current best; already grounded)

TRAIN_VISION_LORA = True    # unchanged since v1.8b
MERGER_MODULES = [
    "visual.merger.linear_fc1", "visual.merger.linear_fc2",
    *[f"visual.deepstack_merger_list.{i}.linear_fc{j}"
      for i in range(3) for j in (1, 2)],
]

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ,
)

base_regex = get_peft_regex(
    model,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
)
MERGER_REGEX = r".*\.visual\.(?:merger|deepstack_merger_list\.\d+)\.linear_fc[12]"
target_modules = (f"(?:{base_regex})|(?:{MERGER_REGEX})"
                  if MERGER_MODE == "lora" else base_regex)

if MERGER_MODE == "full":
    # modules_to_save cannot train a quantized clone (see v1.9c notebook)
    import bitsandbytes as bnb
    _mods = {n: m for n, m in model.named_modules()
             if any(n.endswith(s) for s in MERGER_MODULES)}
    assert len(_mods) == 8, f"merger modules found: {sorted(_mods)}"
    _quantized = [n for n, m in _mods.items()
                  if isinstance(m, (bnb.nn.Linear4bit, bnb.nn.Linear8bitLt))]
    assert not _quantized, (
        f"merger modules are quantized: {_quantized} — reload with these in "
        "llm_int8_skip_modules or load_in_4bit=False before proceeding.")

model = FastVisionModel.get_peft_model(
    model,
    target_modules=target_modules,
    modules_to_save=MERGER_MODULES if MERGER_MODE == "full" else None,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0.0, bias="none", random_state=3407,
)

# creation gate — the run is void unless the adapter matches the KNOB
_tower_lora = [n for n, _ in model.named_parameters()
               if ".visual.blocks." in n and "lora_A" in n]
_merger_lora = [n for n, _ in model.named_parameters()
                if "merger" in n and "lora_A" in n]
_merger_full = [n for n, p in model.named_parameters()
                if "merger" in n and ".modules_to_save." in n
                and n.endswith(".weight") and p.requires_grad]
assert len(_tower_lora) == 108, f"tower adapters: {len(_tower_lora)}/108"
if MERGER_MODE == "frozen":
    assert not _merger_lora and not _merger_full, "merger adapted but knob=frozen"
elif MERGER_MODE == "lora":
    assert len(_merger_lora) == 8 and not _merger_full,         f"merger lora {len(_merger_lora)}/8 — knob=lora not honoured"
else:
    assert len(_merger_full) == 8 and not _merger_lora,         f"merger clones {len(_merger_full)}/8 — knob=full not honoured"
print(f"MERGER_MODE={MERGER_MODE}: tower {len(_tower_lora)}/108, "
      f"merger lora {len(_merger_lora)}, merger full {len(_merger_full)}")

# weights-only warm start (fresh optimizer + schedule)
assert len(WARM_REVISION) == 40, "invalid revision SHA"
from safetensors.torch import load_file
from peft import set_peft_model_state_dict
local = snapshot_download(WARM_CKPT_REPO, revision=WARM_REVISION,
                          allow_patterns="last-checkpoint/adapter_model.safetensors")
_sd = load_file(f"{local}/last-checkpoint/adapter_model.safetensors")
_loaded_merger_lora = any("merger" in k and "lora" in k for k in _sd)
_loaded_merger_full = any("merger" in k and "lora" not in k for k in _sd)
# (modules_to_save saves PLAIN keys — the adapter-name infix is stripped on save)
# the knob must match what the warm checkpoint carries, or trained merger
# weights would be silently dropped on load
if _loaded_merger_full:
    assert MERGER_MODE == "full", "warm ckpt has full merger weights — set MERGER_MODE='full'"
if _loaded_merger_lora:
    assert MERGER_MODE == "lora", "warm ckpt has merger LoRA — set MERGER_MODE='lora'"
if MERGER_MODE == "full":
    # PEFT's modules_to_save loader KeyErrors when the incoming dict lacks the
    # merger keys (any pre-v1.9c warm start). Inject the base weights — the
    # identity start the gate below verifies; setdefault keeps warm-loaded
    # merger weights when the checkpoint carries them.
    for _n, _m in model.named_modules():
        if any(_n.endswith(s) for s in MERGER_MODULES) and hasattr(_m, "modules_to_save"):
            for _pn, _p in _m.original_module.named_parameters():
                _sd.setdefault(f"{_n}.{_pn}", _p.detach().clone())
missing = set_peft_model_state_dict(model, _sd)
print("unexpected keys:", len(getattr(missing, "unexpected_keys", [])))
_wv = [n for n, p in model.named_parameters()
       if ".visual.blocks." in n and "lora_B" in n and p.detach().abs().max().item() > 0]
assert len(_wv) == 108, f"warm start vision adapter not loaded: {len(_wv)}/108 nonzero"
if MERGER_MODE == "lora" and not _loaded_merger_lora:
    _mz = [n for n, p in model.named_parameters()
           if "merger" in n and "lora_B" in n and p.detach().abs().max().item() > 0]
    assert not _mz, f"merger lora_B nonzero at start: {_mz[:3]}"
if MERGER_MODE == "full":
    _wrappers = [(n, m) for n, m in model.named_modules()
                 if any(n.endswith(s) for s in MERGER_MODULES)
                 and hasattr(m, "modules_to_save")]
    assert len(_wrappers) == 8, f"modules_to_save wrappers: {len(_wrappers)}/8"
    if not _loaded_merger_full:
        for _n, _w in _wrappers:
            assert torch.allclose(
                _w.modules_to_save["default"].weight.detach().float(),
                _w.original_module.weight.detach().float()),                 f"merger clone differs from base at start: {_n}"
        print("merger clones at identity (fresh full-weight start)")
    else:
        print("merger clones warm-loaded from checkpoint")
print(f"warm start OK from {WARM_CKPT_REPO}@{WARM_REVISION[:8]}")

if TRAIN_VISION_LORA:
    # patch-embed require-grad fix (see v1.8b/v1.9 notebooks for the diagnosis)
    def _vision_embeds_require_grad(module, inputs, output):
        if not torch.is_grad_enabled():
            return output
        output.requires_grad_(True)
        return output
    _patch_embed = next(m for n, m in model.named_modules()
                        if n.endswith("visual.patch_embed"))
    _patch_embed.register_forward_hook(_vision_embeds_require_grad)
    print("vision LoRA gradient fix: ON")

tokenizer.image_processor = AutoImageProcessor.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct", min_pixels=MIN_PIX, max_pixels=MAX_PIX,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable: {trainable/1e6:.1f}M")
print("resolution:", tokenizer.image_processor.size)


In [ ]:
# Cell 4 — conversation format + collator (native res preserved by resize='max')
def to_conversation(sample):
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": sample["image"]},
                {"type": "text", "text": sample["question"]},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": sample["answer"]},
            ]},
        ]
    }

from unsloth.trainer import UnslothVisionDataCollator

collator = UnslothVisionDataCollator(
    model, tokenizer,
    formatting_func=to_conversation,
    resize="max",
    max_seq_length=MAX_SEQ,
    train_on_responses_only=True,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# guardrail: one real batch must show page-res pixels + masked labels
_first_page = next(iter(ktiv_pages))   # streamed source: no integer indexing
batch = collator([_first_page, synth3[0]])
pv = batch["pixel_values"]
assert pv is not None and pv.shape[0] > 40000, (
    f"{pv.shape[0]} patch rows — expected >40k for two ~6.5MP images; "
    "the resolution policy is not reaching the collator")
labels = batch["labels"]
unmasked = labels[0][labels[0] != -100]
_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
assert _first_page["answer"][:30] in _tok.decode(unmasked)
print(f"collator OK: pixel rows={pv.shape[0]}, labels masked")

In [ ]:
# Cell 5 — gradient-flow verification (cheap; run BEFORE burning GPU-hours)
# Language LoRA always; tower LoRA iff TRAIN_VISION_LORA; merger per the KNOB.
FastVisionModel.for_training(model)
_vb = {k: (v.to(model.device) if torch.is_tensor(v) else v)
       for k, v in collator([next(iter(ktiv_pages))]).items()}
model(**_vb).loss.backward()
tower_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
           if ".visual.blocks." in n and "lora_B" in n and p.grad is not None]
mrg_lora_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
              if "merger" in n and "lora_B" in n and p.grad is not None]
mrg_full_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
              if "merger" in n and ".modules_to_save." in n and n.endswith(".weight")
              and p.grad is not None]
lang_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
          if ".visual." not in n and "lora_B" in n and p.grad is not None]
model.zero_grad(set_to_none=True)
assert lang_g and max(lang_g) > 0, "language LoRA got no gradients"
if TRAIN_VISION_LORA:
    assert len(tower_g) == 108 and min(tower_g) > 0, (
        f"tower LoRA dead or partial: {len(tower_g)}/108 tensors with grads")
if MERGER_MODE == "lora":
    assert len(mrg_lora_g) == 8 and min(mrg_lora_g) > 0,         f"merger LoRA dead or partial: {len(mrg_lora_g)}/8"
elif MERGER_MODE == "full":
    assert len(mrg_full_g) == 8 and min(mrg_full_g) > 0,         f"merger full weights dead or partial: {len(mrg_full_g)}/8"
else:
    assert not mrg_lora_g and not mrg_full_g, "merger got grads but knob=frozen"
print(f"grad gate OK (MERGER_MODE={MERGER_MODE}): tower min|g|={min(tower_g):.2e} "
      f"language max|g|={max(lang_g):.2e}")


In [ ]:
# Cell 6a — OPTIONAL throughput benchmark (flag-gated; run once, then set False)
# The A100 ran v18 at batch 1x8 with 13.8/40GB VRAM — headroom. This times
# fwd+bwd for candidate batch geometries at the REAL resolution so the long
# run uses the fastest safe config. To compare gradient-checkpointing modes
# ("unsloth" vs True), change it in cell 3 and rerun cells 3-6a once each.
RUN_THROUGHPUT_BENCH = False
BATCH_GEOMETRIES = [(1, 8), (2, 4), (4, 2)]   # (per_device_batch, grad_accum)

if RUN_THROUGHPUT_BENCH:
    import time
    FastVisionModel.for_training(model)
    bench_rows = list(ktiv_pages.take(8)) + [synth3[i] for i in range(8)]
    for bs, ga in BATCH_GEOMETRIES:
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            t0 = time.time(); n_micro = 0
            for step in range(2):                    # 2 optimizer steps
                for micro in range(ga):
                    rows = [bench_rows[(n_micro + i) % len(bench_rows)] for i in range(bs)]
                    b = {k: (v.to(model.device) if torch.is_tensor(v) else v)
                         for k, v in collator(rows).items()}
                    (model(**b).loss / ga).backward()
                    n_micro += 1
                model.zero_grad(set_to_none=True)
            dt = (time.time() - t0) / 2
            peak = torch.cuda.max_memory_allocated() / 1e9
            print(f"batch {bs}x{ga}: {dt:.1f}s/optimizer-step, peak {peak:.1f}GB")
        except torch.cuda.OutOfMemoryError:
            print(f"batch {bs}x{ga}: OOM — skip")
            torch.cuda.empty_cache()
    model.zero_grad(set_to_none=True)
    print("pick the fastest non-OOM geometry and set it in cell 6, then set "
          "RUN_THROUGHPUT_BENCH = False")

In [ ]:
# Cell 6 — v2.1 mixture (grounding 20%, ten families) + per-domain eval + training
import inspect
import wandb
from datasets import concatenate_datasets, interleave_datasets
from huggingface_hub import list_repo_files
from trl import SFTConfig, SFTTrainer
from unsloth import is_bf16_supported

CKPT_REPO = "isaacmg/qwen3-vl-8b-hebrew-v21b-ckpt"  # PUBLIC; NEW repo — v21-ckpt keeps the image-blind run js3ku3tw as evidence;
                                                     # never reuse it or v20a/b/c's (auto-resume would load their last-checkpoint)
OUT_DIR = "outputs_v21b"
wandb.login(key=userdata.get("WANDB_API_KEY"))

# v2.1 mixture: transcription backbone (v2.0a's proportions, rescaled) + grounding at
# 20% split across the ten families (docs/v21_grounding_boxes_design.md).
GROUNDING_SHARE = 0.20
GROUNDING_WEIGHTS = {"locate": 0.16, "locate_word": 0.18, "line_index": 0.14, "line_of_phrase": 0.06,
                     "read_box": 0.08, "read_box_word": 0.08, "grounded_detect": 0.14,
                     "grounded_crop": 0.10, "layout_qa": 0.03, "grounded_page": 0.03}
assert abs(sum(GROUNDING_WEIGHTS.values()) - 1.0) < 1e-9
assert set(GROUNDING_WEIGHTS) == set(GROUNDING_TASKS), "every grounding family needs a weight"
_base = {"ktiv_pages": 0.30, "genizah": 0.19, "synth3": 0.15, "ktiv_regions": 0.07,
         "ktiv_crops": 0.07, "pages_small": 0.09, "gemara_crops": 0.05}   # v2.0a's 0.92
_scale = (1.0 - GROUNDING_SHARE) / sum(_base.values())
_map = {"ktiv_pages": ktiv_pages, "genizah": as_stream(genizah), "synth3": as_stream(synth3),
        "ktiv_regions": ktiv_regions, "pages_small": as_stream(pages_small),
        "gemara_crops": as_stream(gemara_crops)}
sources, _probs, _names = [], [], []
for _name, _p in _base.items():
    if _name == "ktiv_crops":   # v2.0a drew section+line crops from one pool: keep their row ratio
        _ns, _nl = N_ROWS["section_transcribe"], N_ROWS["line_transcribe"]
        sources += [ktiv_sections, ktiv_lines]
        _probs += [_p * _scale * _ns / (_ns + _nl), _p * _scale * _nl / (_ns + _nl)]
        _names += ["ktiv_sections", "ktiv_lines"]
    else:
        sources.append(_map[_name]); _probs.append(_p * _scale); _names.append(_name)
for _name in GROUNDING_TASKS:
    sources.append(grounding_streams[_name]); _probs.append(GROUNDING_SHARE * GROUNDING_WEIGHTS[_name])
    _names.append(_name)
assert abs(sum(_probs) - 1.0) < 1e-9, f"mixture sums to {sum(_probs)}"
print("mixture:", {n: round(p, 4) for n, p in zip(_names, _probs)})
# all sources are IterableDatasets (streamed families + as_stream'd small sets);
# all_exhausted restarts a drained source, so short families never end the run.
mixture = interleave_datasets(sources, probabilities=_probs, seed=3407,
                              stopping_strategy="all_exhausted")
# val loss every 100 steps across every domain INCLUDING the new grounding families
def _val(task, n):
    return ktiv3_val.filter(lambda t: t == task, input_columns="task").select(range(n))
eval_ds = concatenate_datasets([
    _val("fragment_transcribe", 40),
    genizah_val.select(range(40)),
    _val("region_transcribe", 15),
    talmud_val.filter(lambda t: t == "page_extract", input_columns="task")
              .select(range(30)),
    talmud_val.filter(
        lambda t, s: t == "crop_transcribe" and s == "gemara",
        input_columns=["task", "section"]).select(range(15)),
    synth3_eval.select(range(20)),
    _val("locate", 15), _val("read_box", 10), _val("layout_qa", 10),
    _val("locate_word", 15), _val("line_index", 10),
    _val("grounded_detect", 5), _val("grounded_crop", 5),
])

def make_sft_config(**kw):
    params = inspect.signature(SFTConfig.__init__).parameters
    if "max_seq_length" in kw and "max_seq_length" not in params:
        kw["max_length"] = kw.pop("max_seq_length")
    dropped = {k: kw.pop(k) for k in list(kw) if k not in params}
    if dropped:
        print(f"⚠️ dropped unsupported SFTConfig kwargs: {sorted(dropped)}")
    return SFTConfig(**kw)

def make_trainer(**kw):
    try:
        return SFTTrainer(**kw)
    except TypeError as e:
        if "tokenizer" in kw and ("tokenizer" in str(e) or "processing_class" in str(e)):
            kw["processing_class"] = kw.pop("tokenizer")
            return SFTTrainer(**kw)
        raise

resume_dir = None
try:
    # hub_strategy="checkpoint" pushes a rolling "last-checkpoint/" folder
    files = list_repo_files(CKPT_REPO)
    if any(f.startswith("last-checkpoint/") for f in files):
        snapshot_download(CKPT_REPO, allow_patterns="last-checkpoint/*",
                          local_dir=OUT_DIR)
        resume_dir = f"{OUT_DIR}/last-checkpoint"
        print("resuming from last-checkpoint")
except Exception as e:
    print(f"no checkpoint repo yet ({type(e).__name__}) — fresh start")

FastVisionModel.for_training(model)
trainer = make_trainer(
    model=model, tokenizer=tokenizer, data_collator=collator,
    train_dataset=mixture, eval_dataset=eval_ds,
    args=make_sft_config(
        # 1x8 proven to fit (~16/40GB, ~65 s/step, compute-bound). To use the idle VRAM
        # for more throughput at the SAME effective batch (8): run cell 6a
        # (RUN_THROUGHPUT_BENCH=True) to confirm 2x4 fits, then set 2,4 here. Don't
        # switch blind — 2x micro-batch ~doubles activation memory and may OOM.
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        dataloader_num_workers=2, dataloader_pin_memory=True,  # overlap shard I/O with compute
        max_steps=2000,                    # kill-anytime, resumes
        learning_rate=5e-5,                # if MERGER_MODE="full" and
                                           # eval@100 > 0.82: restart at 2e-5
        warmup_ratio=0.02, lr_scheduler_type="cosine", weight_decay=0.01,
        logging_steps=10,
        eval_strategy="steps", eval_steps=100, per_device_eval_batch_size=1,
        save_steps=100, save_total_limit=2,
        push_to_hub=True, hub_model_id=CKPT_REPO,
        hub_strategy="checkpoint", hub_private_repo=False,
        optim="adamw_8bit", seed=3407, output_dir=OUT_DIR,
        report_to="wandb", run_name="genizah_v21b",
        bf16=is_bf16_supported(), fp16=not is_bf16_supported(),
        remove_unused_columns=False, dataset_text_field="",
        # STREAMED train set ⇒ Accelerate would default to DataLoaderDispatcher, which slices every batch
        # tensor along dim 0 to the inferred batch size (1): Qwen3-VL packs image patches along dim 0, so
        # pixel_values [25728, 1536] became [1, 1536] and run js3ku3tw trained image-blind (train loss 3.0
        # from step 10, eval rising). docs/v21_independent_dispatch_findings.md. Gated below.
        accelerator_config={"dispatch_batches": False},
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=MAX_SEQ,
    ),
)
# NOTE (streamed train set): on resume the Trainer replays and skips the already-seen
# batches (re-downloads that share of the shards, ~10 min per 1,000 steps) — expected.
# ---- DATALOADER GATE (2026-09-11, js3ku3tw post-mortem) — checks the PREPARED train dataloader, not the collator ----
assert trainer.args.accelerator_config.dispatch_batches is False, "dispatch_batches=False did not reach the trainer"
_dl = trainer.get_train_dataloader()
assert "Dispatcher" not in type(_dl).__name__, f"train dataloader is {type(_dl).__name__} — batches would be truncated"
_it = iter(_dl)
for _k in range(2):
    _b = next(_it)
    _need = int(_b["image_grid_thw"].prod(dim=-1).sum())
    assert _b["pixel_values"].shape[0] == _need, (
        f"batch {_k}: pixel_values has {_b['pixel_values'].shape[0]} patch rows, grid needs {_need} — truncated")
    if _k == 0:
        _first = _b
# the loss on a prepared batch must equal the loss on the same row collated directly (image intact end to end)
_row = next(iter(mixture))                     # same seed ⇒ the same first row the dataloader yields
_direct = {k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in collator([_row]).items()}
assert torch.equal(_direct["input_ids"], _first["input_ids"]), (
    "first dataloader batch is not the first mixture row — rerun the cell; if it persists the mixture is non-deterministic")
model.eval()
with torch.no_grad():
    _lp = model(**{k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in _first.items()}).loss.item()
    _ld = model(**_direct).loss.item()
assert abs(_lp - _ld) <= 0.05 * max(1.0, _ld), f"prepared-dataloader loss {_lp:.3f} != direct-collate loss {_ld:.3f}"
_eb = next(iter(trainer.get_eval_dataloader()))
assert _eb["pixel_values"].shape[0] == int(_eb["image_grid_thw"].prod(dim=-1).sum()), "eval batch truncated"
print(f"dataloader gate OK: {type(_dl).__name__}, patches intact, loss prepared={_lp:.3f} direct={_ld:.3f}")
del _dl, _it, _b, _first, _row, _direct, _eb
FastVisionModel.for_training(model)
if resume_dir:
    # Resume WITHOUT the Trainer's data replay.  With an IterableDataset the default
    # (ignore_data_skip=False) "skips" the first global_step*8 batches by iterating the
    # DataLoader — 9,600 image collations on the CPU while the GPU idles and Colab sees
    # no output for hours (both 09-14 resume attempts died that way).  Instead: resume the
    # optimizer/scheduler/step from the checkpoint and advance the STREAM itself, which
    # streams the bytes but decodes nothing (same seed ⇒ same order ⇒ the unseen tail).
    import json as _json
    _state = _json.load(open(f"{resume_dir}/trainer_state.json"))
    _seen = int(_state["global_step"]) * trainer.args.gradient_accumulation_steps \
        * trainer.args.per_device_train_batch_size
    trainer.args.ignore_data_skip = True
    trainer.train_dataset = mixture.skip(_seen)
    print(f"resume: global_step {_state['global_step']} -> stream skip of {_seen} samples (no collator replay)")
trainer.train(resume_from_checkpoint=resume_dir)

In [ ]:
# Cell 7 — export merged (policy-carrying) model, gated per the KNOB
MERGED_REPO = "isaacmg/qwen3-vl-8b-hebrew-v21-merged"
if TRAIN_VISION_LORA:
    _nz = [n for n, p in model.named_parameters()
           if ".visual.blocks." in n and "lora_B" in n and p.detach().abs().max().item() > 0]
    assert len(_nz) == 108, f"shipped adapter tower lora_B nonzero: {len(_nz)}/108"
if MERGER_MODE == "lora":
    _mnz = [n for n, p in model.named_parameters()
            if "merger" in n and "lora_B" in n and p.detach().abs().max().item() > 0]
    assert len(_mnz) == 8, f"merger lora_B nonzero: {len(_mnz)}/8 — merger never moved"
elif MERGER_MODE == "full":
    _moved = []
    for _n, _m in model.named_modules():
        if any(_n.endswith(s) for s in MERGER_MODULES) and hasattr(_m, "modules_to_save"):
            _c = _m.modules_to_save["default"].weight.detach().float()
            _o = _m.original_module.weight.detach().float()
            if not torch.allclose(_c, _o):
                _moved.append(_n)
    assert len(_moved) == 8, f"merger full weights moved: {len(_moved)}/8"
print(f"shipped checks OK (MERGER_MODE={MERGER_MODE})")
model.save_pretrained_merged("v21-merged", tokenizer, save_method="merged_16bit")
# ship the TRAINING resolution policy with the model (hard rule)
tokenizer.image_processor.save_pretrained("v21-merged")
model.push_to_hub_merged(MERGED_REPO, tokenizer, save_method="merged_16bit", private=True)
print("pushed", MERGED_REPO)
